In [7]:
library(tidyverse)

In [8]:
cv_results <- read_csv(
    "ml_algorithms/results/svm_paper_cv.csv"
)

New names:
• `` -> `...1`
Rows: 7 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): ...1
dbl (4): Cef, Cip, Mer, Tob

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [9]:
cv_results

...1,Cef,Cip,Mer,Tob
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
genexp,0.7910599,0.8119587,0.8746823,0.8838510
genexp+snps,0.7395416,0.8266246,0.8684215,0.8897964
gpa,0.6799746,0.7811176,0.8683959,0.9071392
genexp+gpa,0.7696583,0.8179570,0.8703233,0.8903317
genexp+gpa+snps,0.7703086,0.8006709,0.8544422,0.8972458
gpa+snps,0.6529846,0.8949791,0.8726403,0.8932643
snps,0.6593780,0.8825963,0.5803940,0.8857142


## Analyze data on different data type combinations from the paper

In [4]:
performances <- read_csv(
    "./other_data/perf_all.csv"
)

Rows: 280 Columns: 18
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (3): drug, cv_mode, mode
dbl (15): pos_recall, pos_recall_std, neg_recall, neg_recall_std, precision,...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [36]:
head(performances)

drug,pos_recall,pos_recall_std,neg_recall,neg_recall_std,precision,precision_std,npv,npv_std,balanced_accuracy,balanced_accuracy_std,F1-score_micro,F1-score_micro_std,F1-score_macro,F1-score_macro_std,AUC,cv_mode,mode
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
Meropenem,0.826,0.109,0.843,0.170,0.920,0.081,0.688,0.118,0.834,0.100,0.870,0.085,0.814,0.090,0.877,block_cv,expr
Tobramycin,0.647,0.390,0.852,0.175,0.667,0.361,0.841,0.210,0.750,0.129,0.657,0.306,0.752,0.112,0.823,block_cv,expr
Ciprofloxacin,0.679,0.266,0.859,0.123,0.857,0.213,0.683,0.155,0.769,0.156,0.758,0.229,0.760,0.164,0.798,block_cv,expr
Ceftazidim,0.782,0.210,0.815,0.240,0.806,0.129,0.791,0.233,0.798,0.126,0.794,0.147,0.798,0.129,0.862,block_cv,expr
Meropenem,0.759,0.140,0.820,0.210,0.902,0.102,0.608,0.173,0.790,0.129,0.825,0.099,0.762,0.120,0.883,block_cv,expr_snps
Tobramycin,0.608,0.408,0.901,0.310,0.738,0.413,0.834,0.318,0.755,0.181,0.667,0.358,0.767,0.209,0.803,block_cv,expr_snps


In [5]:
performances_filtered <- performances %>%
    filter(cv_mode == "standard_cv") %>%
    filter(drug == "Tobramycin") %>%
    select(c("drug", "F1-score_macro", "F1-score_macro_std", "mode") )
colnames(performances_filtered) <- c("drug", "F1_score_macro", "F1_score_macro_std", "mode")

In [6]:
performances_filtered %>%
    group_by(mode) %>%
    summarise(F1_score = mean(F1_score_macro))

mode,F1_score
<chr>,<dbl>
expr,0.8800
expr_snps,0.8830
gpa,0.9008
gpa_expr,0.9162
gpa_expr_snps,0.9010
gpa_snps,0.8926
snps,0.8806


# Logistic regression
## C = 1

Table of results: C = 1

Remember the definitions:
* precision s: ts / ts + fs
* recall s: ts / ts + fr
* precision r: tr / tr + fr
* recall r: tr / tr + fs

Comments: if C = 1, too many features are used (between 15000 and 22000 with coefficients higher than 0.01), so data risk to be overfitted. Better decrease C to have a stronger regularization. Same happens with 0.1, because l2 regularization is the Ridge regularization, which is useless in these cases (does not avoid overfit, it seems, but you should check using the performance on the training set if you want to write in the final paper).

The Lasso regression with the 'saga' solver seems to take too much time.

In [20]:
library(tidyverse)
log_reg_scores <- read_csv(
    "ml_algorithms/results/log_reg.csv"
)

New names:
• `` -> `...1`
Rows: 12 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (2): drug, features
dbl (5): ...1, precision_s, precision_r, recall_s, recall_r

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [21]:
log_reg_scores

...1,drug,features,precision_s,precision_r,recall_s,recall_r
<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
0,Cef,genexp,0.7058824,0.6969697,0.7058824,0.6969697
1,Cef,gpa,0.6551724,0.6052632,0.5588235,0.6969697
2,Cef,snps,0.7000000,0.6486486,0.6176471,0.7272727
3,Cip,genexp,0.6756757,0.8000000,0.7812500,0.7000000
4,Cip,gpa,0.6111111,0.7222222,0.6875000,0.6500000
5,Cip,snps,0.6153846,0.7575758,0.7500000,0.6250000
6,Mer,genexp,0.7647059,0.8333333,0.5909091,0.9183673
7,Mer,gpa,0.5625000,0.7636364,0.4090909,0.8571429
8,Mer,snps,0.3125000,0.6909091,0.2272727,0.7755102


In [22]:
log_reg_coefficients <- read_csv(
    "ml_algorithms/results/log_reg_coefficients.csv"
)

New names:
• `` -> `...1`
Rows: 4 Columns: 94268
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (94268): ...1, PA0263.1,tRNA-Arg, PA0296-PA0297,P1, PA0826.2,ssrA, PA083...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [23]:
log_reg_coefficients <- as_tibble(t(log_reg_coefficients))
colnames(log_reg_coefficients) <- c("Cef", "Cip", "Mer", "Tob")
log_reg_coefficients <- log_reg_coefficients %>%
    slice(-1)

In [25]:
log_reg_coefficients %>%
    select(Mer) %>%
    filter(abs(Mer) >= 0.01)

Mer
<dbl>
-0.01059697
0.01021963
0.03006846
0.01125831
0.02784473
-0.02767340
0.01813623
-0.02749689
-0.02473497


## Logistic regression with dimensionality reduction